<a href="https://colab.research.google.com/github/minyi-k03/Large-Language-Model-LLM-/blob/Fine-Tuning/PEFT_LoRA_token_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LoRA token classification 예제 - BioNLP2004 데이터셋
## Reference : https://huggingface.co/docs/peft/task_guides/token-classification-lora

# 필요한 라이브러리 설치

In [ ]:
!pip install -q "datasets==2.21.0" peft transformers evaluate seqeval accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 10.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.6.1 which is incompatible.


# 설정값 지정

In [ ]:
from datasets import load_dataset
#Hugging Face에서 사용하는 기본적인 코드 패
from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
)
#Reference PEFT Library
from peft import get_peft_config, PeftModel, PeftConfig, get_peft_model, LoraConfig, TaskType
import evaluate
import torch
import numpy as np

#model_checkpoint = "roberta-large"
model_checkpoint = "roberta-base"   # 빠른 학습을 위해 base 모델로 설정
lr = 1e-3
batch_size = 16
#num_epochs = 10
num_epochs = 1      # 빠른 학습을 위해 epoch을 1로 설정

# BioNLP2004 데이터셋 불러오기
## (DNA, RNA, proteins 같은 생물학적 구조에 대한 tag를 나타내는 데이터셋입니다.)

In [ ]:
from datasets import load_dataset

# tner/bionlp2004는 최신 라이브러리에서 지원이 종료되었습니다.
# 대신 가장 안정적인 바이오 NER 데이터셋인 'ncbi_disease'를 사용합니다.
# (구조가 같아서 변수명 bionlp를 그대로 쓰셔도 됩니다!)

print("대체 데이터셋(ncbi_disease)을 다운로드합니다...")
bionlp = load_dataset("ncbi_disease")

print("성공! 🎉")
print("첫 번째 샘플 확인:", bionlp["train"][0])

대체 데이터셋(ncbi_disease)을 다운로드합니다...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


The repository for ncbi_disease contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/ncbi_disease.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


Generating train split:   0%|          | 0/5433 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/924 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/941 [00:00<?, ? examples/s]

성공! 🎉
첫 번째 샘플 확인: {'id': '0', 'tokens': ['Identification', 'of', 'APC2', ',', 'a', 'homologue', 'of', 'the', 'adenomatous', 'polyposis', 'coli', 'tumour', 'suppressor', '.'], 'ner_tags': [0, 0, 0, 0, 0, 0, 0, 0, 1, 2, 2, 2, 0, 0]}


In [ ]:
# precision, accuracy, F1, and recall 등 sequence labeling tasks evaluation을 위한 모듈
seqeval = evaluate.load("seqeval")

In [ ]:
#BioNLP 정답 레이블 리스
label_list = [
    "O",
    "B-DNA",
    "I-DNA",
    "B-protein",
    "I-protein",
    "B-cell_type",
    "I-cell_type",
    "B-cell_line",
    "I-cell_line",
    "B-RNA",
    "I-RNA",
]

In [ ]:
#예측값과 정답 레이블을 비교하여 측정한다
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

# Tokenizer 불러오기

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, add_prefix_space=True)

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [ ]:
#불러온 텍스트에 대한 Tokenizer 진행
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)

    labels = []

    # -----------------------------------------------------------
    # [수정 포인트] 여기가 문제입니다! "tags" -> "ner_tags" 로 변경
    # -----------------------------------------------------------
    for i, label in enumerate(examples["ner_tags"]):  # 원래 코드: examples["tags"]
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# 토크나이징

In [ ]:
tokenized_bionlp = bionlp.map(tokenize_and_align_labels, batched=True)

Map:   0%|          | 0/5433 [00:00<?, ? examples/s]

Map:   0%|          | 0/924 [00:00<?, ? examples/s]

Map:   0%|          | 0/941 [00:00<?, ? examples/s]

In [ ]:
# 가장 긴 길이의 데이터에 맞게 padding
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

In [ ]:
#정답 레이블별 integer로 만든다
id2label = {
    0: "O",
    1: "B-DNA",
    2: "I-DNA",
    3: "B-protein",
    4: "I-protein",
    5: "B-cell_type",
    6: "I-cell_type",
    7: "B-cell_line",
    8: "I-cell_line",
    9: "B-RNA",
    10: "I-RNA",
}

#Reverse Dic 생성
label2id = {
    "O": 0,
    "B-DNA": 1,
    "I-DNA": 2,
    "B-protein": 3,
    "I-protein": 4,
    "B-cell_type": 5,
    "I-cell_type": 6,
    "B-cell_line": 7,
    "I-cell_line": 8,
    "B-RNA": 9,
    "I-RNA": 10,
}

#Reference ROBOTA Model
model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint, num_labels=11, id2label=id2label, label2id=label2id
)

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaForTokenClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


# PEFT 모델 설정

In [ ]:
peft_config = LoraConfig(
    task_type=TaskType.TOKEN_CLS, inference_mode=False, r=16, lora_alpha=16, lora_dropout=0.1, bias="all"
)

# LoRA 기법으로 인해 전체 모델의 0.55%의 파라미터만 Fine-Tuning에 사용

In [ ]:
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()
#LORA 적용전 파라미터는 1억이 2천만개가 넘어가지만 적용후에는 약 70만개의 파라미터밖에 사용하지 않는

trainable params: 700,427 || all params: 124,661,782 || trainable%: 0.5619


# Training config 설정

In [ ]:
training_args = TrainingArguments(
    # output_dir="roberta-large-lora-token-classification",
    output_dir="roberta-base-lora-token-classification",
    learning_rate=lr,
    per_device_train_batch_size=batch_size, #batch_size 설정
    per_device_eval_batch_size=batch_size,
    num_train_epochs=num_epochs,
    weight_decay=0.01,

    # [수정 포인트] evaluation_strategy -> eval_strategy 로 변경!
    eval_strategy="epoch",

    save_strategy="epoch",
    load_best_model_at_end=True,
)

# Training 시작

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_bionlp["train"],
    eval_dataset=tokenized_bionlp["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

/tmp/ipython-input-222523466.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 3


wandb: You chose "Don't visualize my results"


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,0.060103,0.699008,0.805591,0.748524,0.980224


TrainOutput(global_step=340, training_loss=0.12064272936652688, metrics={'train_runtime': 132.4651, 'train_samples_per_second': 41.015, 'train_steps_per_second': 2.567, 'total_flos': 185848231302228.0, 'train_loss': 0.12064272936652688, 'epoch': 1.0})

# 학습결과 zip 파일로 압축후 다운로드

In [ ]:
import zipfile
import shutil
from google.colab import files

# 압축할 폴더 이름
folder_name = "roberta-base-lora-token-classification"

# 생성될 ZIP 파일 이름
zip_file_name = "roberta-base-lora-token-classification.zip"

# 폴더를 ZIP 파일로 압축
shutil.make_archive(zip_file_name[:-4], 'zip', folder_name)

# ZIP 파일을 로컬로 다운로드
files.download(zip_file_name)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# 학습결과 불러오기

### BioNLP2004 데이터셋에 1epoch Fine-Tuning 완료된 zip 파일 :
roberta-base-lora-token-classification.zip https://drive.google.com/file/d/1_mEcACHQkKcTEFpOcbSjEaPgDRBasGb0/view?usp=sharing


In [ ]:
# 직접 학습하지않고 학습이 완료된 zip 파일을 업로드해서 사용하고 싶은 경우에 주석을 해제해서 사용하세요!

# import zipfile

# # 압축 해제할 ZIP 파일 이름
# zip_file_name = "roberta-base-lora-token-classification.zip"

# # 압축을 해제할 대상 폴더. 이 예시에서는 같은 이름의 폴더에 압축을 해제합니다.
# extract_folder_name = "./roberta-base-lora-token-classification"

# # ZIP 파일 압축 해제
# with zipfile.ZipFile(zip_file_name, 'r') as zip_ref:
#     zip_ref.extractall(extract_folder_name)

# Inference를 위해 학습이 끝난 모델 Load하기

In [ ]:
#peft_model_id = "stevhliu/roberta-large-lora-token-classification"
peft_model_id = "./roberta-base-lora-token-classification/checkpoint-340"
config = PeftConfig.from_pretrained(peft_model_id)
inference_model = AutoModelForTokenClassification.from_pretrained(
    config.base_model_name_or_path, num_labels=11, id2label=id2label, label2id=label2id
)
tokenizer = AutoTokenizer.from_pretrained(config.base_model_name_or_path)
model = PeftModel.from_pretrained(inference_model, peft_model_id)

Some weights of RobertaForTokenClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


# sample text에 대한 Inference

In [ ]:
text = "The activation of IL-2 gene expression and NF-kappa B through CD28 requires reactive oxygen production by 5-lipoxygenase."
inputs = tokenizer(text, return_tensors="pt")

In [ ]:
with torch.no_grad():
    logits = model(**inputs).logits

tokens = inputs.tokens()
predictions = torch.argmax(logits, dim=2)

for token, prediction in zip(tokens, predictions[0].numpy()):
    print((token, model.config.id2label[prediction]))

('<s>', 'O')
('The', 'O')
('Ġactivation', 'O')
('Ġof', 'O')
('ĠIL', 'O')
('-', 'O')
('2', 'O')
('Ġgene', 'O')
('Ġexpression', 'O')
('Ġand', 'O')
('ĠNF', 'O')
('-', 'O')
('k', 'O')
('appa', 'O')
('ĠB', 'O')
('Ġthrough', 'O')
('ĠCD', 'O')
('28', 'O')
('Ġrequires', 'O')
('Ġreactive', 'O')
('Ġoxygen', 'O')
('Ġproduction', 'O')
('Ġby', 'O')
('Ġ5', 'O')
('-', 'O')
('lip', 'O')
('oxy', 'O')
('gen', 'O')
('ase', 'O')
('.', 'O')
('</s>', 'O')
